### A Dynamic Graph is a graph where the connections (edges) or nodes (vertices) can change over time, instead of staying fixed.
### In graph neural networks such as DGCNN, the graph is rebuilt at every layer, which is why it is called dynamic.
| Static Graph                        | Dynamic Graph                            |
| ----------------------------------- | ---------------------------------------- |
| Graph is built once before training | Graph is rebuilt during the forward pass |
| Neighbors never change              | Neighbors can change at every layer      |
| Based on original coordinates       | Based on learned features                |
| Used in GraphCNN, GCN               | Used in DGCNN                            |

### How DGCNN implements a dynamic graph

```txt
Input features
      │
      ▼
Compute k-NN
      │
      ▼
Construct graph
      │
      ▼
EdgeConv
      │
      ▼
New features
      │
      ▼
Compute k-NN again
      │
      ▼
Construct a new graph
```

![](images/PointNet_1.png)
# PointNet Architecture Overview

The figure illustrates the original **PointNet** architecture, which consists of two main components: the **Classification Network** (upper blue section) and the **Segmentation Network** (lower yellow section). While both networks share the same feature extraction backbone, they differ in their output layers according to the target task.

## Classification Network

### 1. Input Point Cloud (n × 3)

The input consists of a point cloud containing **n** points, where each point is represented by its three-dimensional Cartesian coordinates **(x, y, z)**. Consequently, the input tensor has a shape of **n × 3**. Since a point cloud is an **unordered set**, the network must be invariant to any permutation of the input points.

### 2. Input Transformation Network (T-Net, 3 × 3)

The first T-Net learns a **3 × 3 affine transformation matrix** that aligns the input point cloud into a canonical coordinate system. This learned transformation improves robustness to geometric variations such as rotation and translation. The transformation matrix is multiplied with the input coordinates before feature extraction.

### 3. Shared Multi-Layer Perceptron (64, 64)

Each point is independently processed using a **shared Multi-Layer Perceptron (MLP)**, meaning that the same network weights are applied to every point. This operation is equivalent to applying a sequence of **1 × 1 convolutions**. The output feature representation has a dimension of **n × 64**.

### 4. Feature Transformation Network (T-Net, 64 × 64)

A second T-Net is applied in the feature space to learn a **64 × 64 transformation matrix**. This feature transformation aligns the learned point features, making them more consistent before higher-level feature extraction. An additional **orthogonality regularization loss** is incorporated during training to encourage the transformation matrix to remain close to an orthogonal matrix, thereby preventing excessive distortion.

### 5. Shared MLP (64, 128, 1024)

The aligned point features are further processed through another shared MLP, progressively increasing the feature dimension to **1024** for each point. The resulting feature tensor has a shape of **n × 1024**.

### 6. Global Feature Extraction via Max Pooling

A **symmetric max-pooling operation** is applied across all points to aggregate the point-wise features into a single **1024-dimensional global feature vector**. Since max pooling is permutation invariant, the resulting global representation remains unchanged regardless of the order of the input points.

### 7. Classification Head

The global feature vector is passed through a sequence of fully connected layers with dimensions **512**, **256**, and finally **k**, where **k** denotes the number of object categories. The final output consists of class scores used for object classification.

---

## Segmentation Network

Unlike object classification, semantic segmentation requires a prediction for **every individual point** in the point cloud. Therefore, both local and global information are utilized.

The intermediate **64-dimensional point features**, obtained after the feature transformation stage, are concatenated with the replicated **1024-dimensional global feature vector**. As a result, each point receives a combined **1088-dimensional feature vector (64 + 1024)**.

These enriched point features are subsequently processed through a shared MLP with output dimensions **512**, **256**, and **128**. Finally, another shared MLP maps each point to **m** output scores, where **m** represents the number of segmentation classes. Consequently, the network produces an output tensor of shape **n × m**, assigning a semantic label to every point.

---

# Key Architectural Insights

### Permutation Invariance

Since point clouds do not possess any inherent ordering, PointNet employs a **symmetric max-pooling operation** to aggregate point-wise features. This design guarantees that the network output remains unchanged regardless of the ordering of the input points.

### Integration of Local and Global Information

For segmentation, PointNet combines **point-level local features** with the **global feature representation**. This enables every point to incorporate both its individual geometric characteristics and the overall context of the object.

### Spatial Alignment Using T-Net

Two transformation networks are incorporated to improve robustness against geometric variations. The first aligns the raw input coordinates, while the second aligns intermediate feature representations. Together, these modules enhance the network's ability to learn transformation-invariant features.

---

# Limitation of PointNet

Although PointNet demonstrates excellent performance for point cloud classification, its primary limitation is that it processes each point **independently** before performing global feature aggregation. As a result, it does **not explicitly capture local neighborhood relationships or geometric structures** among nearby points.

This limitation motivated the development of subsequent architectures such as **PointNet++**, which introduces hierarchical local neighborhood grouping and feature learning, and **DGCNN**, which dynamically constructs graphs to model local geometric relationships throughout the network.
